# CIFAR-10 Starter: Cache + Train/Test/Validation Split

This notebook downloads CIFAR-10 once, caches 10,000 images locally, and then splits the cached dataset into train/test/validation sets.

Environment note: this project is managed with `uv` (`uv sync` to install/update dependencies).

## 1) Imports and Paths
This section imports required libraries and defines all data/cache paths used in the notebook.

In [10]:
# Import libraries and define reusable project paths.
from pathlib import Path
import tarfile
import pickle

import numpy as np

CIFAR10_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
CACHE_DIR = DATA_DIR / "cache"

ARCHIVE_PATH = RAW_DIR / "cifar-10-python.tar.gz"
EXTRACTED_DIR = RAW_DIR / "cifar-10-batches-py"
CACHE_PATH = CACHE_DIR / "cifar10_10000.npz"

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

RAW_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## 2) Helper Functions
This section provides safe extraction, dataset download, and CIFAR batch loading utilities.

In [11]:
# Define reusable helpers for safe extraction and batch loading.
def _safe_extract(tar: tarfile.TarFile, path: Path) -> None:
    base = path.resolve()
    for member in tar.getmembers():
        member_path = (path / member.name).resolve()
        if not str(member_path).startswith(str(base)):
            raise RuntimeError(f"Unsafe path in tar file: {member.name}")
    tar.extractall(path=path)


def download_and_extract_cifar10() -> None:
    import urllib.request

    if EXTRACTED_DIR.exists():
        print(f"Dataset already extracted at: {EXTRACTED_DIR}")
        return

    if not ARCHIVE_PATH.exists():
        print("Downloading CIFAR-10 archive...")
        urllib.request.urlretrieve(CIFAR10_URL, ARCHIVE_PATH)
        print(f"Saved archive to: {ARCHIVE_PATH}")
    else:
        print(f"Archive already exists at: {ARCHIVE_PATH}")

    print("Extracting archive...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as tar:
        _safe_extract(tar, RAW_DIR)
    print(f"Extracted to: {EXTRACTED_DIR}")


def load_cifar_batch(batch_file: Path):
    with batch_file.open("rb") as f:
        batch = pickle.load(f, encoding="bytes")

    images = batch[b"data"]
    labels = np.array(batch[b"labels"], dtype=np.int64)

    images = images.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return images, labels


def split_dataset(images: np.ndarray, labels: np.ndarray, train_ratio: float, test_ratio: float, val_ratio: float, seed: int = 42):
    ratio_sum = train_ratio + test_ratio + val_ratio
    if not np.isclose(ratio_sum, 1.0):
        raise ValueError(f"train_ratio + test_ratio + val_ratio must be 1.0, got {ratio_sum:.4f}")

    n = len(images)
    rng = np.random.default_rng(seed)
    indices = np.arange(n)
    rng.shuffle(indices)

    train_end = int(n * train_ratio)
    test_end = train_end + int(n * test_ratio)

    train_idx = indices[:train_end]
    test_idx = indices[train_end:test_end]
    val_idx = indices[test_end:]

    return {
        "X_train": images[train_idx],
        "y_train": labels[train_idx],
        "X_test": images[test_idx],
        "y_test": labels[test_idx],
        "X_val": images[val_idx],
        "y_val": labels[val_idx],
    }

## 3) Build/Load Cache and Split Data
This section loads cached CIFAR-10 data (or builds it once), then splits the 10,000 images into train/test/validation sets using configurable percentages.

In [12]:
# Build or load 10,000-image cache, then split into train/test/validation sets.
TRAIN_RATIO = 0.7  # X%
TEST_RATIO = 0.2   # Y%
VAL_RATIO = 0.1    # Z%
SEED = 42

if CACHE_PATH.exists():
    print(f"Loading cached dataset from: {CACHE_PATH}")
    with np.load(CACHE_PATH) as cached:
        images = cached["images"]
        labels = cached["labels"]
        class_names = cached["class_names"].tolist()
else:
    print("Cache not found. Building cache from CIFAR-10...")
    download_and_extract_cifar10()

    # data_batch_1 contains exactly 10,000 training images
    batch_file = EXTRACTED_DIR / "data_batch_1"
    images, labels = load_cifar_batch(batch_file)
    class_names = CLASS_NAMES

    np.savez_compressed(
        CACHE_PATH,
        images=images.astype(np.uint8),
        labels=labels.astype(np.int64),
        class_names=np.array(class_names),
    )
    print(f"Cache saved to: {CACHE_PATH}")

splits = split_dataset(images, labels, TRAIN_RATIO, TEST_RATIO, VAL_RATIO, seed=SEED)

X_train, y_train = splits["X_train"], splits["y_train"]
X_test, y_test = splits["X_test"], splits["y_test"]
X_val, y_val = splits["X_val"], splits["y_val"]

print("Done.")
print(f"images shape: {images.shape}")
print(f"labels shape: {labels.shape}")
print(f"unique classes: {len(set(labels.tolist()))}")
print()
print(f"Train set: {X_train.shape[0]} samples ({TRAIN_RATIO:.0%})")
print(f"Test set: {X_test.shape[0]} samples ({TEST_RATIO:.0%})")
print(f"Validation set: {X_val.shape[0]} samples ({VAL_RATIO:.0%})")

Loading cached dataset from: data\cache\cifar10_10000.npz
Done.
images shape: (10000, 32, 32, 3)
labels shape: (10000,)
unique classes: 10

Train set: 7000 samples (70%)
Test set: 2000 samples (20%)
Validation set: 1000 samples (10%)


## 4) Quick Sanity Check
This section verifies split labels and shows a quick peek at the first few labels from each split.

In [13]:
# Validate class labels and inspect first labels from each data split.
print("Train labels (first 10):", y_train[:10].tolist())
print("Test labels (first 10):", y_test[:10].tolist())
print("Validation labels (first 10):", y_val[:10].tolist())
print("First train label name:", class_names[int(y_train[0])])

Train labels (first 10): [0, 0, 9, 2, 7, 8, 9, 3, 0, 1]
Test labels (first 10): [3, 2, 0, 7, 6, 1, 5, 4, 0, 8]
Validation labels (first 10): [0, 9, 7, 0, 0, 0, 3, 8, 7, 6]
First train label name: airplane


## 5) CNN Basics: Sequential, Conv2D, and Pooling
This section explains the core CNN building blocks before implementation.

- Sequential: a linear stack of layers where output from one layer goes directly into the next.
- Conv2D: learns image features (edges, textures, shapes) using small filters such as 3x3.
- MaxPooling2D: reduces feature map size (for example 32x32 to 16x16) while keeping strong signals.

In practice, CNN blocks are often `Conv2D -> MaxPooling2D`, repeated several times, then followed by classification layers.

## 6) Build a Base CNN Architecture
This section creates a simple baseline CNN model for CIFAR-10 classification.

In [14]:
# Build a baseline CNN model using Sequential, Conv2D, and MaxPooling2D.
import tensorflow as tf

num_classes = len(class_names)

model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 356,810 (1.36 MB)

 Trainable params: 356,810 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

## 7) Train the CNN
This section normalizes image pixels, trains the model, and monitors validation performance to reduce overfitting risk.

In [15]:
# Normalize images and train with early stopping + learning-rate scheduling.
X_train_norm = X_train.astype("float32") / 255.0
X_val_norm = X_val.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=6,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-5,
    ),
]

history = model.fit(
    X_train_norm,
    y_train,
    validation_data=(X_val_norm, y_val),
    epochs=30,
    batch_size=64,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/30
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - accuracy: 0.2199 - loss: 2.0970 - val_accuracy: 0.3730 - val_loss: 1.8351 - learning_rate: 0.0010
Epoch 2/30
110/110 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.3556 - loss: 1.7610 - val_accuracy: 0.4530 - val_loss: 1.5239 - learning_rate: 0.0010
Epoch 3/30
110/110 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.4296 - loss: 1.5434 - val_accuracy: 0.4930 - val_loss: 1.3902 - learning_rate: 0.0010
Epoch 4/30
110/110 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.4901 - loss: 1.4122 - val_accuracy: 0.5230 - val_loss: 1.3203 - learning_rate: 0.0010
Epoch 5/30
110/110 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.5163 - loss: 1.3278 - val_accuracy: 0.5610 - val_loss: 1.2486 - learning_rate: 0.0010
Epoch 6/30
110/110 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - accuracy: 0.5473 - loss: 1.2542 - val_accuracy: 0.5730 - val_loss: 1.1676 - learning_rate: 0.0010
Epoch 7/30
110/110 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.5700 - loss: 1.

## 8) Evaluate and 85%+ Accuracy Summary
This section evaluates training and testing accuracy, then reports whether each meets the 85% target.

In [16]:
# Evaluate train/test metrics and print a pass/fail summary against 85% target.
TARGET_ACC = 0.85

train_loss, train_acc = model.evaluate(X_train_norm, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test_norm, y_test, verbose=0)
val_loss, val_acc = model.evaluate(X_val_norm, y_val, verbose=0)

results = {
    "Train": train_acc,
    "Validation": val_acc,
    "Test": test_acc,
}

print("Accuracy Summary (Target: 85%+)")
print("-" * 36)
for split_name, acc in results.items():
    status = "PASS" if acc >= TARGET_ACC else "BELOW TARGET"
    print(f"{split_name:<11}: {acc * 100:6.2f}%  [{status}]")

print("\nLoss Summary")
print("-" * 36)
print(f"Train loss     : {train_loss:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Test loss      : {test_loss:.4f}")

if train_acc >= TARGET_ACC and test_acc >= TARGET_ACC:
    print("\nResult: Train and Test are both above 85%.")
else:
    print("\nResult: At least one metric is below 85%. Tune model/epochs/augmentation and retrain.")

Accuracy Summary (Target: 85%+)
------------------------------------
Train      :  94.59%  [PASS]
Validation :  66.40%  [BELOW TARGET]
Test       :  63.60%  [BELOW TARGET]

Loss Summary
------------------------------------
Train loss     : 0.2046
Validation loss: 1.1443
Test loss      : 1.2523

Result: At least one metric is below 85%. Tune model/epochs/augmentation and retrain.
